# 23 — Datos para la página pública: ¿cuánto va a tardar mi caso?
La página pública (`despliegue/sitio/`) responde la pregunta de quien tiene un caso: **«en mi tipo de proceso y en mi
ciudad, ¿cuánto va a tardar?»**. La persona elige materia, tipo de proceso y lugar; la página muestra:

1. **El tiempo estimado para un caso que entra ahora**, con el modelo de 5 variables del notebook 22 y lo que se sabe
   al empezar el año siguiente al Anuario (2024): la fila que espera es lo que quedó **pendiente al cierre de 2023**, los
   juzgados son los mismos y, como los ingresos de 2024 todavía no se conocen, se usan los de 2023.
2. **Un rango probable** (la mitad central de los errores del modelo en procesos de su tamaño, notebook 22).
3. **Lo que tardaba la fila en 2023**, medido sin modelo: `pendientes al cierre ÷ resueltas × 365` (notebook 10).
4. **La historia:** cómo cambió el tiempo en el país de 2007 a 2023 y cómo cambiaron las resoluciones de su materia.

Este notebook arma ese escenario, calcula la estimación con el modelo guardado (`models/modelo_5var.joblib`) y escribe
`despliegue/sitio/datos/procesos.json`. La página recalcula cada estimación con los árboles del JSON del modelo y la
compara con la de Python.

In [1]:
%run -i modulos/comun.ipynb
%run -i modulos/modelado.ipynb
import joblib

procesos = cargar_procesos()
final = joblib.load(RAIZ / "models" / "modelo_5var.joblib")
TOP5 = json.loads((MODELO_DIR / "top5.json").read_text(encoding="utf-8"))["variables"]
MODELO_JSON = json.loads((RAIZ / "despliegue" / "sitio" / "modelo" / "modelo_5var.json").read_text(encoding="utf-8"))
bandas = pd.read_csv(MODELO_DIR / "bandas.csv")
print(len(procesos), "procesos con movimiento en 2023; variables del modelo:", TOP5)

1240 procesos con movimiento en 2023; variables del modelo: ['log_stock_inicial', 'presion_stock', 'log_ingresos', 'ingresos_por_juzgado', 'stock_por_juzgado']


## El escenario: un caso que entra al empezar 2024
Se cambian solo las entradas: **stock heredado ← pendientes al cierre de 2023**; ingresos y juzgados, los de 2023. Las
variables de ingeniería se recalculan con las mismas fórmulas (`construir_variables`).

In [2]:
base = procesos[["materia_homologada", "tipo_proceso", "territorio", "departamento_derivado", "ambito",
                 "pendientes_inicio_efectivo", "ingresos_totales", "resueltas", "pendientes_fin",
                 "num_juzgados_pagina", "atendidas"]].copy()
for c in ["pendientes_inicio_efectivo", "ingresos_totales", "resueltas", "pendientes_fin", "num_juzgados_pagina",
          "atendidas"]:
    base[c] = base[c].astype("float64")
base["materia_homologada"] = base["materia_homologada"].astype("object")

siguiente = base[["ingresos_totales", "num_juzgados_pagina"]].copy()
siguiente["pendientes_inicio_efectivo"] = base["pendientes_fin"]
siguiente = pd.concat([siguiente, construir_variables(siguiente.assign(
    personal_items_acefalias=np.nan, personal_items_total=np.nan, recibidas_excusa_recusacion=np.nan))], axis=1)
log_est = final.predict(siguiente[TOP5])
base["log_estimado"] = log_est
base["dias_estimado"] = np.clip(np.expm1(log_est), 0, TOPE_DIAS)
base["causas_2024"] = base["pendientes_fin"] + base["ingresos_totales"]      # tamaño del proceso en el escenario
clase = np.clip(np.searchsorted(bandas["desde"].to_numpy(), base["causas_2024"].to_numpy(), side="right") - 1, 0, None)
base["dias_desde"] = np.clip(np.expm1(log_est + bandas["q25"].to_numpy()[clase]), 0, TOPE_DIAS)
base["dias_hasta"] = np.clip(np.expm1(log_est + bandas["q75"].to_numpy()[clase]), 0, TOPE_DIAS)
base["dias_2023"] = tiempo_dias(base["pendientes_fin"], base["resueltas"])
base["pocas_causas"] = base["atendidas"] < MIN_CAUSAS
print("mediana del tiempo estimado para 2024:", round(base["dias_estimado"].median()), "días;",
      "en procesos de 10 o más causas:", round(base.loc[~base["pocas_causas"], "dias_estimado"].median()))
print(int(base["pocas_causas"].sum()), "procesos con menos de", MIN_CAUSAS, "causas (la página avisa que el dato es débil)")

mediana del tiempo estimado para 2024: 288 días; en procesos de 10 o más causas: 269
422 procesos con menos de 10 causas (la página avisa que el dato es débil)


## ¿Se parece la estimación a lo que tardaba la fila en 2023?
La historia (notebook 19) dice que, para el país, el tiempo de un año anticipa el del siguiente con un error mediano de
9 %. Si la estimación del modelo para 2024 se alejara mucho de lo medido en 2023 en los procesos grandes, algo andaría
mal. Se compara en los procesos con tiempo medido.

In [3]:
medido = base["dias_2023"].notna() & ~base["pocas_causas"]
razon = np.log1p(base.loc[medido, "dias_estimado"]) - np.log1p(base.loc[medido, "dias_2023"])
grande = medido & (base["atendidas"] > 500)
razon_g = np.log1p(base.loc[grande, "dias_estimado"]) - np.log1p(base.loc[grande, "dias_2023"])
comparacion = {
    "procesos": int(medido.sum()),
    "correlacion_log": float(np.corrcoef(np.log1p(base.loc[medido, "dias_estimado"]), np.log1p(base.loc[medido, "dias_2023"]))[0, 1]),
    "dentro_doble": float((razon.abs() <= np.log(2)).mean()),
    "dentro_doble_grandes": float((razon_g.abs() <= np.log(2)).mean()), "procesos_grandes": int(grande.sum()),
    "dentro_rango": float(((base.loc[medido, "dias_2023"] >= base.loc[medido, "dias_desde"])
                           & (base.loc[medido, "dias_2023"] <= base.loc[medido, "dias_hasta"])).mean()),
}
print({k: round(v, 3) if isinstance(v, float) else v for k, v in comparacion.items()})

{'procesos': 808, 'correlacion_log': 0.871, 'dentro_doble': 0.762, 'dentro_doble_grandes': 0.93, 'procesos_grandes': 143, 'dentro_rango': 0.707}


## La historia que muestra la página
La serie nacional, de capitales y de provincias (2007-2023), la comprobación del vaciado y las resoluciones de cada
materia en su ámbito (2020-2023), todo del notebook 19.

In [4]:
tiempos = pd.read_csv(MODELO_DIR / "historia_tiempos.csv")
vaciado = pd.read_csv(MODELO_DIR / "historia_vaciado.csv")
formas = pd.read_csv(MODELO_DIR / "historia_formas.csv", index_col=0)
hm = pd.read_csv(MODELO_DIR / "historia_materias.csv")
historia = {
    "anios": tiempos["gestion"].astype(int).tolist(),
    "tiempo": {a: [round(float(v)) for v in tiempos[a]] for a in ["nacional", "capital", "provincia"]},
    "vaciado_nacional": {str(int(r.gestion)): round(float(r.tiempo_real_vaciado))
                         for r in vaciado[vaciado["ambito"] == "nacional"].itertuples()},
    "error_vaciado_mediano_pct": round(float(vaciado["error_pct"].abs().median()), 1),
    "comprobaciones_vaciado": int(len(vaciado)),
    "error_anio_anterior_pct": round(float(formas.loc["el del año anterior", "error_abs_mediano_pct"]), 1),
    "peor_anio_anterior_pct": round(float(formas.loc["el del año anterior", "peor_pct"]), 1),
}
print({k: v for k, v in historia.items() if not isinstance(v, (dict, list))})

{'error_vaciado_mediano_pct': 11.7, 'comprobaciones_vaciado': 47, 'error_anio_anterior_pct': 8.8, 'peor_anio_anterior_pct': 35.2}


## Nombres para la gente y exportación
Las materias y los lugares se escriben como los conoce cualquiera («Familia», «El Alto», «Provincias de La Paz») y los
tipos de proceso pasan de mayúsculas a oración. Cada fila del JSON es una lista corta para que el archivo pese poco.

In [5]:
MATERIA_CORTA = {
    "Público Civil y Comercial": "Civil y comercial",
    "Público de Familia": "Familia",
    "Público Niñez y Adolescencia": "Niñez y adolescencia",
    "Partido de Trabajo y Seguridad Social": "Trabajo y seguridad social",
    "Partido Administrativo Coactivo Fiscal y Tributario": "Coactivo fiscal y tributario",
}
CIUDAD = {"SUCRE": "Sucre", "LA PAZ": "La Paz", "EL ALTO": "El Alto", "COCHABAMBA": "Cochabamba", "ORURO": "Oruro",
          "POTOSI": "Potosí", "TARIJA": "Tarija", "SANTA CRUZ": "Santa Cruz", "TRINIDAD": "Trinidad", "COBIJA": "Cobija"}
ACRONIMOS = {"afps": "AFP", "rejaps": "REJAPS", "art.": "Art."}


def lugar(r):
    if r["ambito"] == "capital":
        return CIUDAD[r["territorio"]] + (" (ciudad)" if r["territorio"] in ("LA PAZ", "SANTA CRUZ", "COCHABAMBA",
                                                                                 "ORURO", "POTOSI", "TARIJA") else "")
    return "Provincias de " + r["departamento_derivado"]


def tipo_legible(t):
    t = " ".join(t.split()).replace(" -", " – ").replace(" - ", " – ")
    palabras = [ACRONIMOS.get(p.lower(), p.lower()) for p in t.split(" ")]
    s = " ".join(palabras)
    return s[0].upper() + s[1:]


base["lugar"] = base.apply(lugar, axis=1)
base["materia"] = base["materia_homologada"].map(MATERIA_CORTA)
base["tipo"] = base["tipo_proceso"].map(tipo_legible)
assert base["materia"].notna().all()
assert not base.duplicated(["materia", "tipo", "lugar"]).any(), "dos filas con la misma consulta"

materias = list(MATERIA_CORTA.values())
lugares_df = (base[["lugar", "departamento_derivado", "ambito"]].drop_duplicates()
              .sort_values(["departamento_derivado", "ambito", "lugar"]))
lugares = lugares_df["lugar"].tolist()
tipos = (base[["materia", "tipo"]].drop_duplicates().assign(m=lambda d: d["materia"].map(materias.index))
         .sort_values(["m", "tipo"]))
tipo_id = {(r.materia, r.tipo): k for k, r in enumerate(tipos.itertuples())}

# Resueltas de la materia en su ámbito, 2020-2023: la página las muestra como «el ritmo de tu materia».
hm["materia"] = hm["materia_homologada"].map(MATERIA_CORTA)
ritmo = {f"{r.ambito}|{r.materia}": [int(r._3), int(r._4), int(r._5), int(r._6)]
         for r in hm[["ambito", "materia", "2020", "2021", "2022", "2023"]].itertuples()}

filas = []
for r in base.itertuples():
    filas.append([materias.index(r.materia), tipo_id[(r.materia, r.tipo)], lugares.index(r.lugar),
                  int(r.pendientes_inicio_efectivo), int(r.ingresos_totales), int(r.resueltas), int(r.pendientes_fin),
                  int(r.num_juzgados_pagina), round(float(r.log_estimado), 12)])

salida = {
    "descripcion": "Procesos no penales del Anuario Estadístico Judicial 2023 con movimiento en el año, para la consulta "
                   "de la página pública. Generado por notebooks/23_sitio_publico.ipynb.",
    "anio_datos": 2023, "anio_proyeccion": 2024, "min_causas": MIN_CAUSAS, "tope_dias": TOPE_DIAS,
    "materias": [{"nombre": m, "original": o} for o, m in MATERIA_CORTA.items()],
    "lugares": [{"nombre": r.lugar, "departamento": r.departamento_derivado, "ambito": r.ambito}
                for r in lugares_df.itertuples()],
    "tipos": [{"materia": int(r.m), "nombre": r.tipo} for r in tipos.itertuples()],
    "columnas": ["materia", "tipo", "lugar", "stock_inicio", "ingresos", "resueltas", "pendientes_fin", "juzgados",
                 "log_python"],
    "filas": filas,
    "historia": historia,
    "ritmo_materias": {"anios": [2020, 2021, 2022, 2023], "resueltas": ritmo},
    "comparacion_2023": comparacion,
}
ruta = RAIZ / "despliegue" / "sitio" / "datos" / "procesos.json"
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(json.dumps(salida, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
print(len(filas), "filas ·", len(lugares), "lugares ·", len(tipos), "tipos ·", round(ruta.stat().st_size / 1024), "KB")

1240 filas · 19 lugares · 89 tipos · 55 KB


## Control
Lo que la página recibe tiene que reconstruir lo que el modelo vio: los números y la estimación.

In [6]:
J = json.loads(ruta.read_text(encoding="utf-8"))
for k in np.random.default_rng(SEMILLA).choice(len(base), 50, replace=False):
    f = J["filas"][k]
    e = pd.DataFrame([{"ingresos_totales": float(f[4]), "num_juzgados_pagina": float(f[7]),
                       "pendientes_inicio_efectivo": float(f[6])}])
    e = pd.concat([e, construir_variables(e.assign(personal_items_acefalias=np.nan, personal_items_total=np.nan,
                                                   recibidas_excusa_recusacion=np.nan))], axis=1)
    assert abs(final.predict(e[TOP5])[0] - f[8]) < 1e-9
    assert J["tipos"][f[1]]["materia"] == f[0]
assert all(f"{l['ambito']}|{m['nombre']}" in J["ritmo_materias"]["resueltas"]
           for f in J["filas"] for l, m in [(J["lugares"][f[2]], J["materias"][f[0]])])
print("controles OK: 50 filas al azar reconstruyen la estimación; toda materia y ámbito tiene su historia")

controles OK: 50 filas al azar reconstruyen la estimación; toda materia y ámbito tiene su historia


In [7]:
# Ejemplos para el informe y la guía: los trámites con más causas.
ejemplos = (base.sort_values("atendidas", ascending=False)
            [["materia", "tipo", "lugar", "atendidas", "pendientes_fin", "resueltas", "dias_2023", "dias_estimado",
              "dias_desde", "dias_hasta"]].head(8))
ejemplos.to_csv(MODELO_DIR / "ejemplos_sitio.csv", index=False)
display(ejemplos.round(0))
sucre = base[(base["lugar"] == "Sucre") & (base["tipo"].str.lower().str.startswith("divorcio"))]
display(sucre[["materia", "tipo", "lugar", "pendientes_inicio_efectivo", "ingresos_totales", "resueltas",
               "pendientes_fin", "num_juzgados_pagina", "dias_2023", "dias_estimado", "dias_desde", "dias_hasta"]].round(1))

,materia,tipo,lugar,atendidas,pendientes_fin,resueltas,dias_2023,dias_estimado,dias_desde,dias_hasta
370,Familia,Asistencia familiar,Santa Cruz (ciudad),9346.0,4293.0,5053.0,310.0,368.0,244.0,532.0
161,Civil y comercial,Ejecutivo,Santa Cruz (ciudad),9240.0,998.0,8242.0,44.0,49.0,32.0,71.0
361,Familia,Divorcio – desvinculacion Art. 207,Santa Cruz (ciudad),6421.0,2036.0,4385.0,169.0,137.0,90.0,198.0
25,Civil y comercial,Ejecutivo,La Paz (ciudad),6221.0,1857.0,4364.0,155.0,145.0,96.0,209.0
476,Trabajo y seguridad social,Infraccion a leyes sociales,Santa Cruz (ciudad),5938.0,3991.0,1947.0,748.0,757.0,502.0,1093.0
71,Civil y comercial,Ejecutivo,Cochabamba (ciudad),5432.0,388.0,5044.0,28.0,33.0,22.0,48.0
627,Familia,Asistencia familiar,Provincias de Santa Cruz,5230.0,3173.0,2057.0,563.0,591.0,392.0,854.0
269,Familia,Asistencia familiar,El Alto,4741.0,1131.0,3610.0,114.0,78.0,51.0,113.0


,materia,tipo,lugar,pendientes_inicio_efectivo,ingresos_totales,resueltas,pendientes_fin,num_juzgados_pagina,dias_2023,dias_estimado,dias_desde,dias_hasta
222,Familia,Divorcio – desvinculacion Art. 207,Sucre,173.0,778.0,784.0,167.0,10.0,77.7,66.9,44.0,97.0


## Lectura
- **La consulta cubre 1.240 procesos en 19 lugares** (las 10 capitales, El Alto incluido, y las provincias de los 9
  departamentos) y 89 combinaciones de materia y tipo de proceso. El archivo pesa 55 KB.
- **Para un caso que entra en 2024 la estimación mediana es de 288 días** (269 en los procesos con 10 o más causas). Es más
  que la mediana de 2023 (221) porque ahora entran también los procesos chicos, y porque el escenario supone los mismos
  ingresos con la fila de cierre de 2023.
- **La estimación se parece a lo medido en 2023 donde hay datos firmes:** en los 808 procesos con tiempo medido, la
  correlación (en log) es 0,87 y el 76 % queda entre la mitad y el doble; en los 143 procesos con más de 500 causas, el
  **93 %**. Cuadra con la historia del notebook 19: de un año al siguiente el tiempo cambia poco.
- **Los procesos chicos** (422 con menos de 10 causas) también reciben una estimación, porque el modelo aprende de los
  procesos parecidos; la página avisa que es menos firme y les da el rango más ancho.
- **Ejemplos:** asistencia familiar en Santa Cruz (la de más causas del país): 310 días en 2023, unos 368 para un caso
  nuevo (entre 244 y 532). Divorcio en Sucre: 78 días en 2023, unos 67 (entre 44 y 97). Infracción a leyes sociales en
  Santa Cruz: 748 días en 2023, unos 757 (entre 502 y 1.093).
- **No está validado entre años a nivel de tipo de proceso.** La historia valida la medida para el país, las capitales
  y las provincias; para saber cuánto acierta la proyección en cada tipo de proceso hace falta un segundo Anuario con el
  mismo detalle (2022 o 2024).